# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sujan-lab-cell/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Signal checks

Before creating the baseline score, I tested two signals to evaluate whether the patterns in the data support potential action signals.

---

### Signal 1: Staleness / Refresh Signal (`days_since_last_update`)

**Hypothesis:** Pages that have not been updated for a longer duration (e.g., 91+ days) exhibit higher observed decline rates compared to fresher pages (0–30 days), providing descriptive support for a content refresh flag.

See the bucket table output in the code cell below.

**Verdict: MIXED**

*Explanation:* Pages unupdated for 91–180 days show an observed decline rate of **61.11%**, compared to **51.14%** for pages updated within 0–30 days (+9.97 percentage points higher decline rate for stale pages). Overall, pages with 91+ days since update have a 60.85% decline rate vs 51.20% for fresher pages (<91 days). However, the oldest bucket (181+ days) has a lower decline rate (47.13%) likely due to a smaller sample size (n=174) or evergreen content. Thus, the verdict is **MIXED**: while staleness does not guarantee decline, 91+ days provides a useful empirical threshold for human review.

---

### Signal 2: Search Position → Click-Through Rate (`avg_position` vs Weighted CTR)

**Hypothesis:** Pages ranking worse in Google search results (higher `avg_position`) receive lower overall click-through rates, supporting FlyRank's CTR-fix and search position logic.

See the bucket table output in the code cell below.

**Verdict: CONFIRMED**

*Explanation:* Using the ML-06 weighted CTR methodology (`total_clicks / total_impressions * 100`), weighted CTR decreases as search position worsens: **0.49%** for Top 3 (1–3), **0.35%** for Page 1 (3.1–10), **0.35%** for Striking Distance (10.1–20), **0.15%** for Pages 3–5 (20.1–50), and **0.04%** for Deep positions (50+). Pages with `avg_position == 0` (1,205 rows) represent missing position data rather than rank zero and are explicitly categorized as "no position data (0)". The weighted CTR pattern confirms that search rank visibility correlates directly with user click behavior.

---

### Section 1 Takeaway

Both signals provide useful descriptive evidence for decision-support:
1. **Staleness (`days_since_last_update >= 91`)** serves as a valid condition for identifying pages that may benefit from a content refresh review.
2. **Search Position / Weighted CTR** complements staleness by demonstrating that worse position rankings directly correspond to lower aggregate CTR.

*Note:* Neither signal is causal or a guaranteed predictor of Google's algorithm. They serve as transparent features for human decision-support.

In [1]:
import pandas as pd
from pathlib import Path

# Load dataset from repository root
DATA_PATH = Path("data/raw/content_refresh_anonymized.csv")
assert DATA_PATH.exists(), "Starter CSV not found — run this notebook from repository root."
df = pd.read_csv(DATA_PATH)

# Derive observed decline label from trend_direction for descriptive audit only
# (This label is never used as an input feature in the baseline score)
df["is_declining_label"] = df["trend_direction"].eq("down").astype(int)

print("==================================================")
print("SIGNAL 1: Staleness / Refresh Signal")
print("==================================================")

# Create staleness buckets
df["staleness_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=[-1, 30, 90, 180, float("inf")],
    labels=["0-30 days", "31-90 days", "91-180 days", "181+ days"]
)

# Calculate page count (n), decline count, and decline rate (%)
signal1_table = (
    df.groupby("staleness_bucket", observed=False)
    .agg(
        n=("content_id", "count"),
        decline_count=("is_declining_label", "sum"),
        decline_rate=("is_declining_label", "mean")
    )
    .reset_index()
    .rename(columns={"staleness_bucket": "bucket"})
)
signal1_table["decline_rate"] = (signal1_table["decline_rate"] * 100).round(2)

print("\nSignal 1 Bucket Table (Staleness vs Observed Decline Rate):")
print(signal1_table.to_string(index=False))
print("\nVerdict: MIXED — 91-180 days shows elevated decline (61.11% vs 51.14% for 0-30 days), providing descriptive support for a refresh flag.")

print("\n==================================================")
print("SIGNAL 2: Search Position vs Weighted CTR")
print("==================================================")

# Create position buckets and handle avg_position == 0 explicitly
pos_df = df.copy()
pos_df["position_bucket"] = pd.cut(
    pos_df["avg_position"],
    bins=[0, 3, 10, 20, 50, float("inf")],
    labels=["top 3 (1-3)", "page 1 (3.1-10)", "striking distance (10.1-20)", "pages 3-5 (20.1-50)", "deep (50+)"],
    include_lowest=False
)
pos_df["position_bucket"] = pos_df["position_bucket"].cat.add_categories(["no position data (0)"])
pos_df.loc[pos_df["avg_position"] == 0, "position_bucket"] = "no position data (0)"

# Calculate page count (n), total_clicks, total_impressions, and weighted_ctr (%)
# weighted_ctr = (total clicks / total impressions) * 100
signal2_table = (
    pos_df.groupby("position_bucket", observed=False)
    .agg(
        n=("content_id", "count"),
        total_clicks=("clicks_90d", "sum"),
        total_impressions=("impressions_90d", "sum")
    )
    .reset_index()
    .rename(columns={"position_bucket": "bucket"})
)
signal2_table["weighted_ctr"] = (
    (signal2_table["total_clicks"] / signal2_table["total_impressions"]) * 100
).round(2)

print("\nSignal 2 Bucket Table (Search Position vs Weighted CTR %):")
print(signal2_table[["bucket", "n", "total_clicks", "total_impressions", "weighted_ctr"]].to_string(index=False))
print("\nVerdict: CONFIRMED — Weighted CTR decreases as search position worsens (0.49% in Top 3 down to 0.04% in Deep positions).")


SIGNAL 1: Staleness / Refresh Signal

Signal 1 Bucket Table (Staleness vs Observed Decline Rate):
     bucket     n  decline_count  decline_rate
  0-30 days 20480          10473         51.14
 31-90 days   175            103         58.86
91-180 days  9171           5604         61.11
  181+ days   174             82         47.13

Verdict: MIXED — 91-180 days shows elevated decline (61.11% vs 51.14% for 0-30 days), providing descriptive support for a refresh flag.

SIGNAL 2: Search Position vs Weighted CTR

Signal 2 Bucket Table (Search Position vs Weighted CTR %):
                     bucket     n  total_clicks  total_impressions  weighted_ctr
                top 3 (1-3)  1141         37042            7560663          0.49
            page 1 (3.1-10) 11842        311928           89361420          0.35
striking distance (10.1-20)  7273         79552           22819980          0.35
        pages 3-5 (20.1-50)  7225         53883           35038692          0.15
                 deep 

## 2. Build the ranked queue

Now that I have checked the two signals, I want to turn the baseline idea into something I can actually use as a review queue.

For this baseline, I am using a simple rule:

> **If clicks in the most recent 30 days are lower than clicks in the previous 30 days (`clicks_last_30d < clicks_prev_30d`), flag the page for review.**

This is the same idea as the FlyRank session's **April clicks < March clicks** rule, adapted to the field names present in the starter dataset (`clicks_last_30d` vs `clicks_prev_30d`).

A drop in clicks alone does not tell me that a page definitely needs to be fixed. It only gives me a reason to look at the page.

After flagging the pages, I rank them by their **most recent 30-day impressions** (`impressions_last_30d`). This puts higher-volume pages earlier in the queue, because a page with more search visibility represents a larger potential area for review.

The score is therefore intentionally simple:

- **Flagged page (`clicks_last_30d < clicks_prev_30d`):** `score = impressions_last_30d`
- **Not flagged (`clicks_last_30d >= clicks_prev_30d`):** `score = 0`

Each page also gets a reason code (`click_decline_high_volume`, `click_decline_lower_volume`, or `no_decline_detected`) and an action label (`refresh_review` vs `monitor`) so that the queue is easy for a human reviewer to understand.

This is a transparent baseline, not a learned model. I will keep it fixed so that later model results can be compared against the same starting point.

The ranked queue is written to `work/outputs/baseline_action_score.csv`.

In [2]:
import pandas as pd
from pathlib import Path

# Load dataset from repository root
DATA_PATH = Path("data/raw/content_refresh_anonymized.csv")
assert DATA_PATH.exists(), "Starter CSV not found — run this notebook from repository root."
df = pd.read_csv(DATA_PATH)

# Rule: Flag page if most recent 30-day clicks are lower than previous 30-day clicks
df["is_flagged"] = df["clicks_last_30d"] < df["clicks_prev_30d"]

# Transparent baseline score: impressions_last_30d for flagged pages, 0 otherwise
df["baseline_score"] = df["impressions_last_30d"].where(df["is_flagged"], 0)

# Assign action label: refresh_review vs monitor
df["action"] = df["is_flagged"].map({True: "refresh_review", False: "monitor"})

# Assign human-readable reason codes
df["reason_code"] = "no_decline_detected"
df.loc[
    df["is_flagged"] & df["impressions_last_30d"].ge(1_000),
    "reason_code"
] = "click_decline_high_volume"
df.loc[
    df["is_flagged"] & df["impressions_last_30d"].lt(1_000),
    "reason_code"
] = "click_decline_lower_volume"

# Build and sort the ranked queue deterministically
# Primary sort: baseline_score (descending)
# Secondary sort: impressions_last_30d (descending)
# Tie-breaker: content_id (ascending)
queue_columns = [
    "client_id", "content_id", "baseline_score", "reason_code", "action",
    "clicks_last_30d", "clicks_prev_30d", "impressions_last_30d"
]

queue = df[queue_columns].copy()
queue = queue.rename(columns={"baseline_score": "score"})

queue = queue.sort_values(
    ["score", "impressions_last_30d", "content_id"],
    ascending=[False, False, True]
).reset_index(drop=True)

# Insert 1-indexed rank column at the start
queue.insert(0, "rank", queue.index + 1)

# Write output to work/outputs/baseline_action_score.csv
OUTPUT_PATH = Path("work/outputs/baseline_action_score.csv")
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
queue.to_csv(OUTPUT_PATH, index=False)

print(f"Successfully generated and wrote {len(queue):,} rows to {OUTPUT_PATH}")
print(f"Flagged pages count: {df['is_flagged'].sum():,} ({df['is_flagged'].mean()*100:.2f}% of total dataset)")
print("\nReason code breakdown:")
print(queue["reason_code"].value_counts().to_string())
print("\nFirst 10 rows of the ranked baseline queue:")
print(queue.head(10).to_string(index=False))


Successfully generated and wrote 30,000 rows to work\outputs\baseline_action_score.csv
Flagged pages count: 6,806 (22.69% of total dataset)

Reason code breakdown:
reason_code
no_decline_detected           23194
click_decline_lower_volume     3787
click_decline_high_volume      3019

First 10 rows of the ranked baseline queue:
 rank         client_id           content_id  score               reason_code         action  clicks_last_30d  clicks_prev_30d  impressions_last_30d
    1 client_19581e27de content_aaef01a50def 170559 click_decline_high_volume refresh_review              435              461                170559
    2 client_6208ef0f77 content_2dba2b1f9536 139891 click_decline_high_volume refresh_review              314              323                139891
    3 client_4e07408562 content_5fe46e04994d 120791 click_decline_high_volume refresh_review              220              250                120791
    4 client_4e07408562 content_9532f197bbc8 109317 click_decline_high_volu

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.